# Multi-Objective Supplier Sourcing with cuOpt Python API

This notebook uses the cuOpt Python API to trace the **efficient frontier** of a sourcing decision — splitting one
order across suppliers when reliability and diversification genuinely conflict, and there's no single best split,
only a curve of optimal tradeoffs.

The most reliable suppliers tend to cluster (same region, same logistics), so demanding higher reliability quietly
pushes the order toward a few correlated names — concentration risk. We trace that tradeoff with the
**ε-constraint method** and read each point's **sensitivity** (the reliability-floor dual):

1. **Two objectives** — minimize concentration risk, maximize reliability.
2. **Keep one as the objective, constrain the other** — minimize concentration subject to `reliability ≥ ε`.
3. **Sweep ε** across the achievable reliability range; each solve is one frontier point.
4. **Read the frontier** — and, because this is a continuous QP, read each point's **dual**: the sensitivity
   d(concentration)/d(reliability), how much diversification one more point of reliability costs.

This is the same recipe as `portfolio_optimization/QP_portfolio_frontier_duals`, on a procurement problem — two
competing objectives and a solver for one of them is all it takes. (This workflow is also packaged as the
`cuopt-multi-objective-exploration` skill.)

## Environment Setup

In [ ]:
import subprocess
import html
from IPython.display import display, HTML

def check_gpu():
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
        result.check_returncode()
        lines = result.stdout.splitlines()
        gpu_info = lines[2] if len(lines) > 2 else "GPU detected"
        gpu_info_escaped = html.escape(gpu_info)
        display(HTML(f"""
        <div style="border:2px solid #4CAF50;padding:10px;border-radius:10px;background:#e8f5e9;">
            <h3>✅ GPU is enabled</h3>
            <pre>{gpu_info_escaped}</pre>
        </div>
        """))
        return True
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, FileNotFoundError, IndexError) as e:
        display(HTML("""
        <div style="border:2px solid red;padding:15px;border-radius:10px;background:#ffeeee;">
            <h3>⚠️ GPU not detected!</h3>
            <p>This notebook requires a <b>GPU runtime</b>.</p>

            <h4>If running in Google Colab:</h4>
            <ol>
              <li>Click on <b>Runtime → Change runtime type</b></li>
              <li>Set <b>Hardware accelerator</b> to <b>GPU</b></li>
              <li>Then click <b>Save</b> and <b>Runtime → Restart runtime</b>.</li>
            </ol>

            <h4>If running in Docker:</h4>
            <ol>
              <li>Ensure you have <b>NVIDIA Docker runtime</b> installed (<code>nvidia-docker2</code>)</li>
              <li>Run container with GPU support: <code>docker run --gpus all ...</code></li>
              <li>Or use: <code>docker run --runtime=nvidia ...</code> for older Docker versions</li>
              <li>Verify GPU access: <code>docker run --gpus all nvidia/cuda:12.0.0-base-ubuntu22.04 nvidia-smi</code></li>
            </ol>

            <p><b>Additional resources:</b></p>
            <ul>
              <li><a href="https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/install-guide.html" target="_blank">NVIDIA Container Toolkit Installation Guide</a></li>
            </ul>
        </div>
        """))
        return False

check_gpu()

In [ ]:
# Uncomment for your CUDA version if cuOpt is not already installed (e.g., Google Colab):
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu12  # CUDA 12
# !pip install --upgrade --extra-index-url https://pypi.nvidia.com cuopt-cu13  # CUDA 13

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cuopt.linear_programming.problem import Problem, QuadraticExpression, MINIMIZE
print("Imports ready (cuOpt QP solver)")

## Step 1 — the suppliers and their two objectives

A simulated panel of 12 suppliers across 3 regions. Each has a **reliability** score (to maximize) and a **unit
cost**; the reliable suppliers are pricier and cluster in one region. The **concentration risk** is a quadratic
form `wᵀ C w` over the allocation `w`: `C` has a high within-region correlation and near-zero across regions, so
piling the order into one region—even across different suppliers there—reads as concentrated. Minimizing it is
what pushes the order to spread across regions (multi-sourcing).

In [ ]:
np.random.seed(7)

regions = ["A", "B", "C"]
region_of = [r for r in regions for _ in range(4)]          # 12 suppliers, 4 per region
suppliers = [f"{r}{i+1}" for r in regions for i in range(4)]
n = len(suppliers)

# Reliable suppliers cluster in region A (and cost more); region C is cheap but less reliable.
base_rel = {"A": 0.95, "B": 0.85, "C": 0.75}
base_cost = {"A": 12.0, "B": 9.0, "C": 6.0}
reliability = np.array([np.clip(base_rel[r] + np.random.normal(0, 0.02), 0.5, 0.99) for r in region_of])
unit_cost = np.array([max(2.0, base_cost[r] + np.random.normal(0, 1.0)) for r in region_of])

# Concentration matrix: 1.0 on the diagonal, 0.6 within region, 0.05 across (dense, PSD).
within, across = 0.6, 0.05
concentration = np.array([[1.0 if i == j else (within if region_of[i] == region_of[j] else across)
                           for j in range(n)] for i in range(n)])

summary = pd.DataFrame({"Region": region_of, "Reliability": reliability, "Unit Cost": unit_cost}, index=suppliers)
summary.style.format({"Reliability": "{:.3f}", "Unit Cost": "${:.2f}"})

## Step 2 — minimize concentration risk, capturing the reliability-floor dual

The model splits one order: weights `w` sum to 1, no supplier over a cap, total unit cost within budget. The
objective is the concentration quadratic `wᵀ C w`; the swept ε-constraint is a **reliability floor** whose
**`.DualValue`** we keep after each solve. For a continuous QP that dual is the **sensitivity**
d(concentration)/d(reliability) — the diversification cost of demanding one more point of reliability.

**Building the quadratic — matrix vs term-by-term.** The base portfolio notebook builds its quadratic term by
term, one Python expression per matrix entry (`for i: for j: quad += c * w[i] * w[j]`) — fine for a small dense
matrix, but O(n²) Python objects as it grows. cuOpt's `QuadraticExpression` also takes the matrix **directly**
(`qmatrix=C, qvars=w`), a single vectorized construction. Both express the same `wᵀ C w`; the matrix form is the
one to reach for on a dense `C`. We use it here.

In [ ]:
def solve_min_concentration_qp_dual(concentration, reliability, unit_cost, budget,
                                    target_reliability=None, max_weight=0.22):
    """Solve the min-concentration sourcing QP and return the split plus the reliability-floor dual.

    Minimizes concentration risk (w' * concentration * w) subject to a fully-allocated order
    (weights sum to 1), a per-supplier cap, a unit-cost budget, and — when target_reliability is
    given — a minimum-reliability epsilon-constraint. That constraint's .DualValue is the
    sensitivity d(concentration)/d(reliability), accurate to cuOpt's barrier-solver tolerance.

    Parameters
    ----------
    concentration : ndarray (n, n)
        Concentration-risk matrix (symmetric, positive semidefinite).
    reliability : ndarray (n,)
        Per-supplier reliability scores.
    unit_cost : ndarray (n,)
        Per-supplier unit cost.
    budget : float
        Maximum allowed weighted unit cost (sum of cost_i * w_i).
    target_reliability : float, optional
        Minimum weighted reliability (the swept epsilon-constraint); None = unconstrained.
    max_weight : float
        Upper bound on each supplier's share of the order.

    Returns
    -------
    dict
        {"weights", "concentration", "reliability", "cost", "dual", "active", "status"}.
    """
    n = len(reliability)
    prob = Problem("Supplier_Sourcing")
    w = [prob.addVariable(lb=0.0, ub=float(max_weight), name=f"w_{i}") for i in range(n)]

    # Concentration quadratic w' C w, built from the matrix directly (vs term-by-term).
    quad = QuadraticExpression(qmatrix=concentration, qvars=w)
    prob.setObjective(quad, sense=MINIMIZE)

    prob.addConstraint(sum(w) == 1, name="fully_allocated")
    prob.addConstraint(sum(float(unit_cost[i]) * w[i] for i in range(n)) <= float(budget), name="budget")
    rel_con = None
    if target_reliability is not None:
        rel_expr = sum(float(reliability[i]) * w[i] for i in range(n))
        rel_con = prob.addConstraint(rel_expr >= float(target_reliability), name="min_reliability")

    prob.solve()
    status = prob.Status.name if hasattr(prob.Status, "name") else str(prob.Status)
    weights = np.array([w[i].Value for i in range(n)])
    conc = float(weights @ concentration @ weights)
    dual = abs(float(rel_con.DualValue)) if rel_con is not None else 0.0   # sensitivity d(conc)/d(reliability)
    return {"weights": weights, "concentration": conc, "reliability": float(reliability @ weights),
            "cost": float(unit_cost @ weights), "dual": dual,
            "active": int((weights > 1e-4).sum()), "status": status}

BUDGET = 11.5
base = solve_min_concentration_qp_dual(concentration, reliability, unit_cost, BUDGET)
print(f"Min-concentration split: status={base['status']}, concentration={base['concentration']:.4f}, "
      f"reliability={base['reliability']:.3f}, active suppliers={base['active']}")

## Step 3 — sweep the reliability floor → the frontier (and its duals)

Sweep the reliability floor ε from the unconstrained mix’s reliability up toward the achievable ceiling; each ε is
one standard cuOpt solve. Floors past what the caps and budget allow are simply infeasible — we skip them, exactly
as the portfolio sweep skips non-optimal points.

In [ ]:
rel_min = base["reliability"]
rel_max = float(reliability.max())
targets = np.linspace(rel_min + 0.002, rel_max * 0.999, 25)

rels, concs, duals, actives, flagged = [], [], [], [], 0
for t in targets:
    r = solve_min_concentration_qp_dual(concentration, reliability, unit_cost, BUDGET, target_reliability=t)
    if r["status"] not in ("Optimal", "PrimalFeasible"):
        continue                                   # floor beyond the caps/budget — infeasible, skip
    if r["status"] != "Optimal":
        flagged += 1
    rels.append(r["reliability"]); concs.append(r["concentration"])
    duals.append(r["dual"]); actives.append(r["active"])

rels, concs, duals, actives = map(np.array, (rels, concs, duals, actives))
print(f"Frontier points: {len(rels)} | not certified-Optimal (PrimalFeasible): {flagged}")
print(f"Active suppliers: {actives.max()} (most diversified) -> {actives.min()} (most concentrated) as the floor rises")
print(f"Sensitivity d(concentration)/d(reliability): {duals.min():.3f} -> {duals.max():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sc = axes[0].scatter(rels * 100, concs, c=actives, cmap="viridis", s=60, zorder=3)
axes[0].plot(rels * 100, concs, "-", color="navy", lw=1.0, alpha=0.5, zorder=2)
axes[0].set_xlabel("Required reliability (%)"); axes[0].set_ylabel("Concentration risk  (wᵀ C w)")
axes[0].set_title("Sourcing frontier (concentration vs reliability)"); axes[0].grid(alpha=0.3)
fig.colorbar(sc, ax=axes[0], label="active suppliers")

axes[1].plot(rels * 100, duals, "o-", color="purple", lw=1.6)
axes[1].set_xlabel("Required reliability (%)"); axes[1].set_ylabel("Sensitivity  d(concentration)/d(reliability)")
axes[1].set_title("Marginal diversification cost of reliability (cuOpt QP dual)"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Step 4 — read the frontier

- The **frontier** (left) is the concentration-vs-reliability Pareto set — each point the least-concentrated split
  that meets its reliability floor. There's no single "best"; you choose where to sit. The color shows the order
  spreading across **fewer suppliers** as you demand more reliability — the reliable names cluster, so reliability
  and diversification pull apart.
- The **dual** (right) is the **sensitivity** d(concentration)/d(reliability): the diversification given up for one
  more point of reliability. It steepens along the frontier — the marginal cost of reliability rises, which is
  exactly where a knee analysis pays off.

### Takeaway — reusing this on your own problem
Two competing objectives and a solver for one of them is all you need: keep one objective, turn the other into a
swept constraint (`f₂ ≥ ε` or `≤ ε`), solve across the range, collect the non-dominated points, and — for an LP or
QP — read the constraint's dual for the marginal exchange rate. The **budget** here is a fixed constraint, but it
carries a dual too: re-read `budget`'s `.DualValue` for the diversification bought per dollar of budget. And when
the objective is a dense quadratic, build it from the matrix (`QuadraticExpression(qmatrix=...)`) rather than term
by term.

## License

SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.